# P04 · Rendimiento y análisis de datos PQRS

## Una comparación explicada paso a paso

Este laboratorio está pensado para estudiantes de distintas disciplinas. No presupone experiencia en desarrollo de software. El énfasis está en **formular una comparación justa, leer la evidencia y tomar una decisión**, no en construir una aplicación.

Trabajaremos con los seis cortes semestrales completos de PQRS de 2023, 2024 y 2025. Primero comprobaremos que los datos son equivalentes en Polars y DuckDB. Después compararemos Polars y pandas sobre una tarea idéntica.

**Pregunta central:** ¿qué diferencias de tiempo y memoria observamos en este equipo, con estos datos y con esta operación?


## La misma ruta lógica de las diapositivas

El laboratorio sigue los cinco pasos usados en clase:

```text
1. Formular       2. Modelar       3. Diseñar       4. Ejecutar y validar       5. Interpretar y decidir
la pregunta          el costo          la medición       datos y resultados          con evidencia
```

Dentro del paso 4 avanzaremos de manera incremental: fuentes, un corte, seis cortes, Parquet común y medición. Cada bloque termina con un punto de control.

La lógica es importante: **no medimos velocidad hasta demostrar que los motores procesan la misma población y producen el mismo resultado**.


## Qué aprenderá y qué no necesita programar

Al terminar podrá:

- explicar por qué una comparación necesita datos y reglas equivalentes;
- distinguir preparación de datos de medición de rendimiento;
- interpretar mediana, variación y uso de memoria;
- reconocer límites antes de recomendar una herramienta.

Las celdas llamadas **instrumento suministrado** contienen detalles técnicos de lectura o medición. Se ejecutan sin modificarlas. Las celdas de análisis son cortas y muestran las operaciones que sí deben comprenderse.


## Glosario mínimo

- **Motor analítico:** herramienta que lee y transforma datos, como pandas, Polars o DuckDB.
- **DataFrame:** tabla en memoria formada por filas y columnas.
- **Consulta diferida o lazy:** instrucciones que se preparan primero y se ejecutan cuando se solicita el resultado.
- **Agrupar:** reunir filas que comparten claves para calcular un resumen, por ejemplo un conteo por mes y departamento.
- **Materializar:** ejecutar el trabajo y obtener un resultado concreto.
- **Tiempo de pared:** segundos reales transcurridos según el reloj.
- **RSS:** memoria física aproximada ocupada por el proceso.
- **Mediana:** valor central de las repeticiones ordenadas; evita resumir el experimento con un único caso extremo.


# Paso 1 · Formular la pregunta

## La decisión que motiva el experimento

Un equipo debe producir cada mes un conteo de reportes por periodo, mes y departamento. Los datos ya superan cinco millones de filas. El equipo conoce pandas y quiere saber si conviene incorporar Polars.

La pregunta experimental es:

> ¿Cómo cambian el tiempo y la memoria al resolver la misma consulta con pandas y Polars sobre una entrada Parquet común?

Complete antes de ejecutar:

- **Hipótesis:** espero que ____________________.
- **Razón:** porque ____________________.
- **Evidencia que apoyaría la hipótesis:** ____________________.
- **Evidencia que la refutaría:** ____________________.


# Paso 2 · Modelar el costo

La consulta recorre varias etapas. Una mejora en una etapa pequeña puede no cambiar el tiempo total.

```text
leer bytes  →  decodificar tipos  →  seleccionar columnas  →  agrupar  →  materializar
    I/O              CPU                    I/O + CPU           CPU/RAM       RAM
```

Para este ejercicio esperamos que el formato Parquet reduzca lectura y decodificación. Polars puede aprovechar proyección, ejecución diferida y varios hilos. pandas ofrece una API conocida, pero normalmente materializa la tabla en memoria antes de agrupar. Estas son hipótesis causales, no conclusiones.

Antes de continuar, marque cuál recurso cree que dominará: **disco**, **CPU** o **memoria**. Al final contrastará esa predicción con la evidencia.


# Paso 3 · Diseñar la medición

## Variables y contrato de equivalencia

| Tipo | En este laboratorio |
|---|---|
| Variable independiente | motor: pandas o Polars |
| Variables dependientes | tiempo de pared, RSS y throughput |
| Variables controladas | mismo Parquet, columnas, agrupación, nulos, equipo y frontera temporal |
| Variables contextuales | versiones, hilos, caché, almacenamiento y procesos concurrentes |

El contrato exige las mismas filas, pregunta, salida y frontera. El reloj incluye lectura del Parquet, selección, agrupación y materialización. Excluye descarga, normalización, creación del Parquet, ordenación para comparar y exportación.

La presentación explica cinco repeticiones como protocolo general. P04 usa **un calentamiento y tres repeticiones** por motor para ajustarse al volumen y al tiempo de clase. Se conservan las seis mediciones.

**Actualización del caso:** las diapositivas muestran el alcance anterior de tres cortes y 2.444.766 filas. Este notebook amplía el caso a seis cortes y 5.327.501 filas. La ruta experimental permanece igual. Antes de procesar todo el volumen construiremos un piloto de 3.000 filas, como puente con la demostración de clase.


# Paso 4 · Ejecutar y validar

## Etapa 4A · Preparar y verificar las fuentes

### Preparar el entorno

La siguiente celda carga las bibliotecas. Una biblioteca es un conjunto de herramientas ya construidas: pandas y Polars trabajan con tablas; DuckDB permite consultar con SQL; psutil observa memoria y procesador.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from threading import Event, Thread
import gc
import json
import os
import platform
import shutil
import subprocess
import sys
import time

import duckdb
import pandas as pd
import polars as pl
import psutil
import pyarrow as pa
from IPython.display import display

print('Entorno cargado correctamente.')


In [ ]:
# El notebook puede abrirse desde kit/ o desde kit/Notebooks/.
carpeta_actual = Path.cwd().resolve()
if (carpeta_actual / '00_datos.py').exists():
    KIT = carpeta_actual
elif (carpeta_actual.parent / '00_datos.py').exists():
    KIT = carpeta_actual.parent
else:
    raise RuntimeError('Abra el notebook desde kit/ o kit/Notebooks/.')

DATOS = KIT / 'data' / 'pqrs_completos'
SALIDA = KIT / 'salidas' / 'pqrs_completos_2023_2025'
PARQUET_COMUN = SALIDA / 'pqrs_2023_2025_comun.parquet'
SALIDA.mkdir(parents=True, exist_ok=True)

print('Carpeta del kit:', KIT)
print('Carpeta de datos:', DATOS)
print('Carpeta de resultados:', SALIDA)


### Registrar el entorno antes del experimento

El equipo, las versiones y los hilos ayudan a explicar el resultado. Los registramos antes de procesar datos, como indica el protocolo de las diapositivas.


In [ ]:
espacio = shutil.disk_usage(SALIDA)
configuracion_equipo = {
    'inicio_utc': datetime.now(timezone.utc).isoformat(),
    'sistema': platform.platform(),
    'procesador': platform.processor(),
    'cpu_logicas': psutil.cpu_count(logical=True),
    'cpu_fisicas': psutil.cpu_count(logical=False),
    'ram_GB': round(psutil.virtual_memory().total / 1e9, 2),
    'disco_libre_GB': round(espacio.free / 1e9, 2),
    'python': platform.python_version(),
    'pandas': pd.__version__, 'polars': pl.__version__,
    'duckdb': duckdb.__version__, 'pyarrow': pa.__version__,
    'psutil': psutil.__version__,
    'hilos_polars': pl.thread_pool_size(),
    'POLARS_MAX_THREADS': os.environ.get('POLARS_MAX_THREADS'),
}
display(pd.DataFrame([configuracion_equipo]))


### Conocer las fuentes antes de descargarlas

El manifiesto funciona como una ficha técnica: registra nombre, URL, tamaño, separador, número esperado de filas y huella SHA-256. La huella permite detectar si un archivo quedó incompleto o fue sustituido.

Observe la tabla. No continúe todavía: identifique qué corte usa un separador diferente y cuál llega dentro de un ZIP.


In [ ]:
fuentes_base = json.loads((KIT / 'pqrs_fuentes.json').read_text(encoding='utf-8'))
fuentes_nuevas = json.loads(
    (KIT / 'pqrs_fuentes_adicionales_2023_2025.json').read_text(encoding='utf-8')
)
fuentes = sorted(fuentes_base + fuentes_nuevas, key=lambda f: f['periodo'])

catalogo = pd.DataFrame([{
    'corte': f['periodo'],
    'archivo': f['archivo'],
    'filas_esperadas': f['filas'],
    'columnas': len(f['columnas']),
    'separador': repr(f['separador']),
    'formato_descarga': f.get('contenedor', 'csv'),
    'tamaño_MB': round(f['bytes'] / 1_000_000, 1),
    'url': f['url'],
} for f in fuentes])
display(catalogo)
print('Total esperado:', f"{catalogo['filas_esperadas'].sum():,}", 'registros')


### Descargar de forma explícita

Las URL de 2025 son:

- 2025-I: <https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/289c737231d14a968224150c585d08a3/data>
- 2025-II: <https://mapas.supersalud.gov.co/arcgisportal/sharing/rest/content/items/d36192e86d874296aeaf95cd1a53e904/data>

2025-I llega como CSV. 2025-II llega como ZIP; el kit conserva el ZIP y extrae su TXT como `PQRD_2025_II.csv`. La descarga completa produce aproximadamente 3,830 GB descomprimidos.

La bandera queda en `False` para evitar iniciar varios GB por accidente. Cambie a `True` cuando tenga espacio y conexión estable.


In [ ]:
DESCARGAR = False  # Cambie a True para descargar los seis cortes.

if DESCARGAR:
    subprocess.run(
        [sys.executable, '00_datos.py', '--descargar', 'pqrs-completos'],
        cwd=KIT, check=True
    )
else:
    print('Descarga omitida. Los archivos existentes no fueron modificados.')


### Verificar antes de analizar

Esta celda recalcula tamaños, huellas, cabeceras y filas. Si falla, lea el mensaje y deténgase. No conviene intentar “arreglar” una diferencia ignorándola.


In [ ]:
subprocess.run(
    [sys.executable, '00_datos.py', '--verificar', 'pqrs-completos'],
    cwd=KIT, check=True
)

inventario = pd.DataFrame([{
    'corte': f['periodo'],
    'archivo': f['archivo'],
    'url': f['url'],
    'bytes_locales': (DATOS / f['archivo']).stat().st_size,
    'filas_esperadas': f['filas'],
    'sha256_verificado': f['sha256'],
    'estado': 'verificado',
} for f in fuentes])
inventario.to_csv(SALIDA / 'inventario_fuentes.csv', index=False)
display(inventario[['corte', 'archivo', 'bytes_locales', 'filas_esperadas', 'estado']])


### Punto de control 4A

Antes de continuar, responda en sus notas:

1. ¿Por qué un nombre de archivo correcto no garantiza que su contenido sea correcto?
2. ¿Qué diferencia hay entre el ZIP original de 2025-II y el CSV que analizarán los motores?
3. ¿Los seis cortes suman 5.327.501 registros?


## Etapa 4B · Comprender un solo corte

Antes de combinar millones de filas, observaremos cinco registros de 2025-I. Esta práctica reduce la carga cognitiva: primero entendemos una pieza; después repetimos el patrón.

Un CSV no contiene tipos formales. Por eso, en esta exploración inicial leeremos los campos como texto. Más adelante convertiremos únicamente año y mes.


In [ ]:
fuente_demo = next(f for f in fuentes if f['periodo'] == '2025_I')
archivo_demo = DATOS / fuente_demo['archivo']

muestra_polars = pl.read_csv(
    archivo_demo, separator=fuente_demo['separador'],
    infer_schema=False, n_rows=5
)
display(muestra_polars.select([
    'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'
]).to_pandas())


La operación anterior tiene tres decisiones visibles:

1. usa el separador declarado por la fuente;
2. observa solo cinco filas;
3. muestra únicamente las columnas necesarias para el ejercicio.

Ahora DuckDB observará el mismo archivo. SQL expresa la pregunta en forma declarativa: indicamos qué queremos, no cómo recorrer cada fila.


In [ ]:
ruta_demo_sql = str(archivo_demo).replace("'", "''")
muestra_duckdb = duckdb.sql(f"""
    SELECT periodo, mes, pqr_canal, afec_cod_depto, afec_cod_mpio
    FROM read_csv('{ruta_demo_sql}', delim=';', header=true, all_varchar=true)
    LIMIT 5
""").df()
display(muestra_duckdb)


In [ ]:
filas_polars_demo = (
    pl.scan_csv(archivo_demo, separator=';', infer_schema=False)
    .select(pl.len()).collect(engine='streaming').item()
)
filas_duckdb_demo = duckdb.sql(f"""
    SELECT COUNT(*)
    FROM read_csv('{ruta_demo_sql}', delim=';', header=true, all_varchar=true)
""").fetchone()[0]

print('Filas según Polars:', f'{filas_polars_demo:,}')
print('Filas según DuckDB:', f'{filas_duckdb_demo:,}')
assert filas_polars_demo == filas_duckdb_demo == fuente_demo['filas']


### Punto de control 4B

- ¿Ambos motores reconocieron las mismas columnas en la muestra?
- ¿Coincidió el número total de filas?
- ¿Por qué mostrar cinco filas no reemplaza el conteo completo?

Si las respuestas son claras, podemos generalizar el procedimiento a los seis cortes.


## Etapa 4C · Integrar los seis cortes

### Hacer visibles las diferencias

Los archivos no tienen exactamente la misma forma. 2023-I usa nombres en mayúsculas y 2024-I usa coma. La tabla siguiente documenta las excepciones de manera explícita; así evitamos código que intenta adivinar.


In [ ]:
configuracion = pd.DataFrame([
    ['2023_I',  'PQRD_2023_I.csv',  ';', 'AÑO',     'MES', 'PQR_CANAL', 'AFEC_COD_DEPTO', 'AFEC_COD_MPIO'],
    ['2023_II', 'PQRD_2023_II.csv', ';', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
    ['2024_I',  'PQRD_2024_I.csv',  ',', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
    ['2024_II', 'PQRD_2024_II.csv', ';', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
    ['2025_I',  'PQRD_2025_I.csv',  ';', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
    ['2025_II', 'PQRD_2025_II.csv', ';', 'periodo', 'mes', 'pqr_canal', 'afec_cod_depto', 'afec_cod_mpio'],
], columns=[
    'corte', 'archivo', 'separador', 'col_periodo', 'col_mes',
    'col_canal', 'col_departamento', 'col_municipio'
])
display(configuracion)


### Leer con Polars

La función siguiente es un **instrumento suministrado**. Recibe una fila de configuración, lee su archivo y devuelve seis columnas con nombres comunes. Solo año y mes se convierten a números; los códigos geográficos permanecen como texto para no perder ceros iniciales.

No necesita memorizar la función. Identifique sus tres pasos: leer, seleccionar y renombrar.


In [ ]:
def texto_limpio(nombre):
    return pl.col(nombre).str.strip_chars().replace('', None)

def leer_corte_polars(fila):
    datos = pl.scan_csv(
        DATOS / fila.archivo, separator=fila.separador,
        infer_schema=False, ignore_errors=False
    )
    return datos.select(
        pl.lit(fila.corte).alias('corte_archivo'),
        texto_limpio(fila.col_periodo).cast(pl.Int16, strict=False).alias('periodo'),
        texto_limpio(fila.col_mes).cast(pl.Int8, strict=False).alias('mes'),
        texto_limpio(fila.col_canal).alias('pqr_canal'),
        texto_limpio(fila.col_departamento).alias('afec_cod_depto'),
        texto_limpio(fila.col_municipio).alias('afec_cod_mpio'),
    )

cortes_polars = [leer_corte_polars(fila) for fila in configuracion.itertuples(index=False)]
pqrs_polars = pl.concat(cortes_polars, how='vertical')


### Piloto de 3.000 filas

Tomamos 500 filas de cada corte. El piloto permite comprobar la transformación y la consulta con bajo costo. No se usa para concluir rendimiento y no reemplaza la validación completa.


In [ ]:
piloto_polars = pl.concat([corte.head(500) for corte in cortes_polars]).collect()
piloto_pandas = piloto_polars.to_pandas()
claves_piloto = ['corte_archivo', 'periodo', 'mes', 'afec_cod_depto']

salida_polars = (piloto_polars.group_by(claves_piloto)
    .agg(pl.len().alias('reportes')).to_pandas())
salida_pandas = (piloto_pandas.groupby(claves_piloto, dropna=False)
    .size().rename('reportes').reset_index())

for tabla in (salida_polars, salida_pandas):
    tabla['afec_cod_depto'] = tabla['afec_cod_depto'].fillna('<NULO>').astype(str)
    tabla.sort_values(claves_piloto, inplace=True, ignore_index=True)

pd.testing.assert_frame_equal(salida_polars, salida_pandas, check_dtype=False)
assert len(piloto_pandas) == 3_000
display(piloto_pandas.groupby('corte_archivo').size().rename('filas_piloto'))
print('Piloto validado: pandas y Polars producen el mismo agregado.')


In [ ]:
control_polars = (
    pqrs_polars.group_by('corte_archivo').agg(
        pl.len().alias('filas_polars'),
        pl.col('periodo').min().alias('año_min'),
        pl.col('periodo').max().alias('año_max'),
        pl.col('mes').min().alias('mes_min'),
        pl.col('mes').max().alias('mes_max'),
        pl.col('afec_cod_depto').is_null().sum().alias('departamento_nulo'),
        pl.col('afec_cod_mpio').is_null().sum().alias('municipio_nulo'),
    ).sort('corte_archivo').collect(engine='streaming').to_pandas()
)
display(control_polars)


### Leer con DuckDB

El siguiente **instrumento suministrado** aplica la misma tabla de configuración para crear una vista SQL. Después de ejecutarlo, la consulta de control será corta y legible.


In [ ]:
def nombre_sql(nombre):
    return '"' + nombre.replace('"', '""') + '"'

def texto_sql(nombre):
    return f"NULLIF(TRIM(CAST({nombre_sql(nombre)} AS VARCHAR)), '')"

partes_sql = []
for fila in configuracion.itertuples(index=False):
    ruta = str(DATOS / fila.archivo).replace("'", "''")
    partes_sql.append(f"""
        SELECT '{fila.corte}' AS corte_archivo,
               TRY_CAST({nombre_sql(fila.col_periodo)} AS SMALLINT) AS periodo,
               TRY_CAST({nombre_sql(fila.col_mes)} AS TINYINT) AS mes,
               {texto_sql(fila.col_canal)} AS pqr_canal,
               {texto_sql(fila.col_departamento)} AS afec_cod_depto,
               {texto_sql(fila.col_municipio)} AS afec_cod_mpio
        FROM read_csv('{ruta}', delim='{fila.separador}', header=true, all_varchar=true)
    """)

conexion = duckdb.connect()
conexion.execute('CREATE OR REPLACE TEMP VIEW pqrs_duckdb AS ' + ' UNION ALL '.join(partes_sql))


In [ ]:
control_duckdb = conexion.sql('''
    SELECT corte_archivo, COUNT(*) AS filas_duckdb,
           MIN(periodo) AS año_min_duckdb, MAX(periodo) AS año_max_duckdb,
           MIN(mes) AS mes_min_duckdb, MAX(mes) AS mes_max_duckdb,
           COUNT(*) FILTER (WHERE afec_cod_depto IS NULL) AS departamento_nulo_duckdb,
           COUNT(*) FILTER (WHERE afec_cod_mpio IS NULL) AS municipio_nulo_duckdb
    FROM pqrs_duckdb
    GROUP BY corte_archivo
    ORDER BY corte_archivo
''').df()

esperado = catalogo[['corte', 'filas_esperadas']].rename(columns={'corte': 'corte_archivo'})
control_lectura = (esperado
    .merge(control_polars, on='corte_archivo')
    .merge(control_duckdb, on='corte_archivo'))
display(control_lectura)

assert (control_lectura['filas_esperadas'] == control_lectura['filas_polars']).all()
assert (control_lectura['filas_polars'] == control_lectura['filas_duckdb']).all()
assert (control_lectura['departamento_nulo'] == control_lectura['departamento_nulo_duckdb']).all()
assert (control_lectura['municipio_nulo'] == control_lectura['municipio_nulo_duckdb']).all()
control_lectura.to_csv(SALIDA / 'control_lectura_polars_duckdb.csv', index=False)


### Comparar una salida analítica, no solo el total

Dos motores pueden coincidir en el total y diferir en la distribución. Por eso agruparemos por corte, año, mes y departamento. La comparación revisa **todos los grupos**, incluidos los departamentos nulos.


In [ ]:
CLAVES = ['corte_archivo', 'periodo', 'mes', 'afec_cod_depto']

agregado_polars = (pqrs_polars.group_by(CLAVES)
    .agg(pl.len().alias('reportes'))
    .collect(engine='streaming').to_pandas())

agregado_duckdb = conexion.sql('''
    SELECT corte_archivo, periodo, mes, afec_cod_depto, COUNT(*) AS reportes
    FROM pqrs_duckdb
    GROUP BY corte_archivo, periodo, mes, afec_cod_depto
''').df()

def ordenar_para_comparar(tabla):
    resultado = tabla.copy()
    resultado['afec_cod_depto'] = resultado['afec_cod_depto'].fillna('<NULO>').astype(str)
    return resultado.sort_values(CLAVES).reset_index(drop=True)

agregado_polars = ordenar_para_comparar(agregado_polars)
agregado_duckdb = ordenar_para_comparar(agregado_duckdb)
pd.testing.assert_frame_equal(agregado_polars, agregado_duckdb, check_dtype=False)
print('Equivalencia confirmada en', len(agregado_polars), 'grupos.')
display(agregado_polars.head(10))


In [ ]:
equivalencia_inicial = {
    'equivalentes': True,
    'filas_totales': int(control_lectura['filas_polars'].sum()),
    'grupos_comparados': len(agregado_polars),
    'claves': CLAVES,
}
(SALIDA / 'equivalencia_polars_duckdb.json').write_text(
    json.dumps(equivalencia_inicial, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([equivalencia_inicial]))


### Punto de control 4C

No continúe si falló una aserción. Si todo coincide, explique:

- por qué 2023-I necesitó nombres de columnas diferentes;
- por qué los códigos geográficos se conservaron como texto;
- por qué comparar todos los grupos ofrece más evidencia que comparar solo el total.

**Límite semántico:** la clasificación de motivos cambió el 1 de julio de 2023. Este ejercicio no usa esos campos para comparar tendencias.


## Etapa 4D · Preparar una entrada común

CSV es útil para intercambio, pero no conserva tipos y obliga a interpretar texto en cada lectura. Parquet es columnar: permite leer solo las columnas necesarias.

Crearemos **un único Parquet** a partir de la lectura ya conciliada. Polars y pandas recibirán exactamente el mismo archivo. Esta conversión es preparación y queda fuera del reloj.


In [ ]:
pqrs_polars.sink_parquet(PARQUET_COMUN, compression='zstd')

control_parquet = (pl.scan_parquet(PARQUET_COMUN)
    .group_by('corte_archivo').agg(pl.len().alias('filas'))
    .sort('corte_archivo').collect().to_pandas())

assert int(control_parquet['filas'].sum()) == 5_327_501
assert set(control_parquet['corte_archivo']) == set(configuracion['corte'])
display(control_parquet)
print('Tamaño del Parquet:', round(PARQUET_COMUN.stat().st_size / 1_000_000, 1), 'MB')


### Punto de control 4D

El Parquet debe contener los seis cortes y 5.327.501 filas. Si contiene menos, el benchmark sería rápido por una razón equivocada: estaría procesando menos datos.


## Etapa 4E · Comparar Polars y pandas

### Definir exactamente el mismo trabajo

Cada motor debe:

1. leer cuatro columnas del Parquet;
2. agrupar por corte, año, mes y departamento;
3. contar reportes, incluidos los departamentos nulos;
4. materializar el resultado.

Las dos funciones siguientes están una al lado de la otra para facilitar la lectura. Son las operaciones centrales del laboratorio.


In [ ]:
def consulta_pandas():
    datos = pd.read_parquet(PARQUET_COMUN, columns=CLAVES)
    return (datos.groupby(CLAVES, dropna=False, sort=False)
            .size().rename('reportes').reset_index())

def consulta_polars():
    return (pl.scan_parquet(PARQUET_COMUN).select(CLAVES)
            .group_by(CLAVES).agg(pl.len().alias('reportes'))
            .collect(engine='streaming'))


### El plan anticipa parte del costo

Antes de cronometrar, inspeccionamos el plan de Polars. Busque si aparecen las cuatro columnas seleccionadas y la agregación. Esta evidencia conecta el código con las etapas de costo discutidas en las diapositivas.


In [ ]:
plan_polars = (pl.scan_parquet(PARQUET_COMUN).select(CLAVES)
    .group_by(CLAVES).agg(pl.len().alias('reportes')))
print(plan_polars.explain(optimized=True))


### Calentar y comprobar equivalencia

La primera ejecución carga código y puede poblar cachés. La usaremos como calentamiento y no entrará en el resumen. Al mismo tiempo comprobaremos que ambas funciones producen la misma salida.


In [ ]:
calentamiento_pandas = ordenar_para_comparar(consulta_pandas())
calentamiento_polars = ordenar_para_comparar(consulta_polars().to_pandas())
pd.testing.assert_frame_equal(
    calentamiento_pandas, calentamiento_polars, check_dtype=False
)
assert int(calentamiento_pandas['reportes'].sum()) == 5_327_501
print('Calentamiento terminado. Las salidas son equivalentes.')

del calentamiento_pandas, calentamiento_polars
gc.collect()


### Instrumento suministrado para medir

La función siguiente actúa como cronómetro y observador de memoria. No hace parte de la consulta ni necesita modificarse. Registra tiempo de pared, memoria inicial y pico RSS del proceso de Jupyter.

Esta medición es apropiada para una práctica guiada, pero tiene un límite: ambos motores comparten el mismo kernel y sus bibliotecas pueden retener memoria. En un estudio de producción convendría repetir cada prueba en procesos aislados.


In [ ]:
def medir(motor, operacion, intervalo=0.02):
    proceso = psutil.Process()
    detener = Event()
    observaciones = []

    def observar_memoria():
        while not detener.wait(intervalo):
            observaciones.append(proceso.memory_info().rss)

    gc.collect()
    rss_inicial = proceso.memory_info().rss
    monitor = Thread(target=observar_memoria, daemon=True)
    monitor.start()
    inicio = time.perf_counter()
    resultado = operacion()
    segundos = time.perf_counter() - inicio
    observaciones.append(proceso.memory_info().rss)
    detener.set()
    monitor.join()
    pico = max([rss_inicial] + observaciones)

    medicion = {
        'motor': motor, 'segundos': segundos,
        'rss_inicial_MB': rss_inicial / 1_000_000,
        'rss_pico_MB': pico / 1_000_000,
        'aumento_pico_MB': (pico - rss_inicial) / 1_000_000,
        'intervalo_sondeo_s': intervalo,
    }
    return resultado, medicion


### Realizar tres repeticiones

Alternaremos el orden para que un motor no sea siempre el primero o el segundo. Conservaremos todas las mediciones, incluso si una parece atípica.


In [ ]:
orden = [
    (1, 'pandas'), (1, 'polars'),
    (2, 'polars'), (2, 'pandas'),
    (3, 'pandas'), (3, 'polars'),
]
operaciones = {'pandas': consulta_pandas, 'polars': consulta_polars}
registros = []
resultados_control = {}

for repeticion, motor in orden:
    resultado, medicion = medir(motor, operaciones[motor])
    medicion['repeticion'] = repeticion
    registros.append(medicion)
    if repeticion == 1:
        resultados_control[motor] = resultado
    print(motor, repeticion,
          f"{medicion['segundos']:.3f} s",
          f"{medicion['aumento_pico_MB']:.1f} MB adicionales")
    if repeticion != 1:
        del resultado
    gc.collect()

benchmark = pd.DataFrame(registros)
benchmark.to_csv(SALIDA / 'benchmark_polars_pandas.csv', index=False)
display(benchmark)


In [ ]:
resultado_pandas = ordenar_para_comparar(resultados_control['pandas'])
resultado_polars = ordenar_para_comparar(resultados_control['polars'].to_pandas())
pd.testing.assert_frame_equal(resultado_pandas, resultado_polars, check_dtype=False)

equivalencia_benchmark = {
    'equivalentes': True,
    'filas_entrada': int(resultado_pandas['reportes'].sum()),
    'grupos_comparados': len(resultado_pandas),
}
(SALIDA / 'equivalencia_polars_pandas.json').write_text(
    json.dumps(equivalencia_benchmark, indent=2), encoding='utf-8'
)
display(pd.DataFrame([equivalencia_benchmark]))


### Resumir sin ocultar la variación

La mediana representa el centro de tres observaciones y es menos sensible a una ejecución inusualmente lenta. Mínimo y máximo muestran cuánto variaron las mediciones. Con solo tres repeticiones no buscamos inferencia estadística; buscamos una comparación transparente y repetible.


In [ ]:
resumen = (benchmark.groupby('motor', as_index=False)
    .agg(segundos_mediana=('segundos', 'median'),
         segundos_min=('segundos', 'min'),
         segundos_max=('segundos', 'max'),
         rss_pico_mediana_MB=('rss_pico_MB', 'median'),
         aumento_pico_mediana_MB=('aumento_pico_MB', 'median')))
resumen['filas_por_segundo'] = 5_327_501 / resumen['segundos_mediana']
resumen.to_csv(SALIDA / 'benchmark_polars_pandas_resumen.csv', index=False)
display(resumen)


In [ ]:
configuracion_equipo.update({
    'fin_utc': datetime.now(timezone.utc).isoformat(),
    'metodo_memoria': 'pico RSS del kernel compartido, sondeo cada 0.02 s',
    'frontera_temporal': 'lectura Parquet + selección + agrupación + materialización',
    'calentamientos_por_motor': 1,
    'repeticiones_por_motor': 3,
    'filas_procesadas': 5_327_501,
})
(SALIDA / 'configuracion_benchmark.json').write_text(
    json.dumps(configuracion_equipo, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([configuracion_equipo]))


# Paso 5 · Interpretar y decidir

Complete un párrafo para cada punto:

1. **Validez:** Polars, DuckDB y pandas procesaron ______ filas y los agregados ______.
2. **Tiempo:** la mediana fue ______ para pandas y ______ para Polars. La variación observada fue ______.
3. **Memoria:** el pico o aumento de RSS fue ______. Recuerde que el kernel es compartido.
4. **Explicación:** relacione el resultado con formato columnar, selección de columnas, ejecución diferida, paralelismo y materialización.
5. **Límites:** mencione caché, almacenamiento, procesos concurrentes, tres repeticiones y características de este equipo.
6. **Decisión:** indique en qué situación usaría cada motor y qué prueba adicional realizaría antes de migrar un proceso real.

No escriba “Polars siempre es más rápido” ni “pandas es lento”. La conclusión válida está limitada a este conjunto, esta consulta, estas versiones y este equipo.


## Puente con Spark y la decisión de capacidad

Este notebook no cronometra Spark. Los notebooks 05 y 06 permiten revisar su plan, particiones y costo de arranque. Spark se convierte en candidato cuando el volumen supera la capacidad de un equipo, aparecen uniones o movimientos grandes, o se necesita tolerancia a fallos. Sin una medición equivalente, no afirmamos que Spark sea más rápido o más lento.

La ruta de decisión de la clase se conserva: validar el resultado, reducir datos leídos, mejorar formato y plan, medir otra vez y escalar solo si la restricción persiste.


# Ejercicio para realizar en clase · Decisión tecnológica

**Duración:** 35 minutos, en equipos de dos o tres personas.

**Situación:** una entidad debe generar semanalmente el mismo resumen departamental. Dispone de un equipo con 8 GB de RAM. El volumen podría duplicarse durante el próximo año y el equipo analista conoce pandas, pero no Polars.

Sigan la ruta de las diapositivas:

1. **Formular:** escriban la decisión que debe tomar la entidad y una hipótesis.
2. **Modelar:** indiquen qué etapa parece explicar la diferencia observada.
3. **Diseñar:** identifiquen una variable que mantendrían controlada y una nueva prueba necesaria.
4. **Ejecutar y validar:** usen las seis mediciones y confirmen equivalencia.
5. **Interpretar y decidir:** recomienden pandas, Polars o una adopción gradual. Expliquen costo de aprendizaje, memoria, tiempo y límites.

No se acepta una recomendación basada únicamente en el tiempo más bajo.


In [ ]:
medianas = resumen.set_index('motor')
indicadores_clase = {
    'filas': 5_327_501,
    'equivalencia_confirmada': equivalencia_benchmark['equivalentes'],
    'mediana_pandas_s': float(medianas.loc['pandas', 'segundos_mediana']),
    'mediana_polars_s': float(medianas.loc['polars', 'segundos_mediana']),
    'razon_tiempo_pandas_sobre_polars': float(
        medianas.loc['pandas', 'segundos_mediana'] / medianas.loc['polars', 'segundos_mediana']
    ),
    'throughput_pandas_filas_s': float(medianas.loc['pandas', 'filas_por_segundo']),
    'throughput_polars_filas_s': float(medianas.loc['polars', 'filas_por_segundo']),
    'aumento_rss_pandas_MB': float(medianas.loc['pandas', 'aumento_pico_mediana_MB']),
    'aumento_rss_polars_MB': float(medianas.loc['polars', 'aumento_pico_mediana_MB']),
}
(SALIDA / 'indicadores_ejercicio_clase.json').write_text(
    json.dumps(indicadores_clase, indent=2, ensure_ascii=False), encoding='utf-8'
)
display(pd.DataFrame([indicadores_clase]))


## Respuesta del equipo

Edite esta celda y complete:

- **Decisión que debemos tomar:** …
- **Hipótesis inicial:** …
- **Resultado equivalente que validamos:** …
- **Etapa o recurso que probablemente explica la diferencia:** …
- **Recomendación para la entidad:** …
- **Límite principal de esta evidencia:** …
- **Próxima prueba antes de adoptar o escalar:** …


# Productos de entrega

Entregue el notebook ejecutado y la carpeta `kit/salidas/pqrs_completos_2023_2025/` con:

- `inventario_fuentes.csv`;
- `control_lectura_polars_duckdb.csv`;
- `equivalencia_polars_duckdb.json`;
- `pqrs_2023_2025_comun.parquet`;
- `benchmark_polars_pandas.csv`;
- `benchmark_polars_pandas_resumen.csv`;
- `equivalencia_polars_pandas.json`;
- `configuracion_benchmark.json`;
- `indicadores_ejercicio_clase.json`;
- un informe breve siguiendo la guía de interpretación.

No es necesario adjuntar los CSV originales, pero el inventario debe permitir reconstruir el ejercicio.
